# Camada Bronze - Ingestão de Dados

Este notebook realiza a ingestão dos dados brutos utilizados no MVP de Engenharia de Dados.

## Fontes

- **IBGE/PAM:** dados históricos da produção agrícola de soja.
- **ANP:** dados de produção de biodiesel e matérias-primas utilizadas em sua produção.

Os dados são armazenados inicialmente na camada Bronze, preservando o conteúdo original das fontes antes das etapas de limpeza, padronização e integração.


In [0]:
import requests

# Metadados da tabela 5457 - PAM/IBGE
url_ibge = "https://apisidra.ibge.gov.br/desctabapi.aspx?c=5457"

response_ibge = requests.get(url_ibge, timeout=30)

print("Status:", response_ibge.status_code)
print(response_ibge.text[:5000])

Status: 200


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html xmlns="http://www.w3.org/1999/xhtml">

<head id="Head1"><title>
	Apresenta os códigos das dimensões e suas listas de opções
</title><link id="Link1" rel="stylesheet" type="text/css" href="Content/Estilo.css" />
<script type="text/javascript" language="JavaScript"> 
<!--

// Abre a janela para a lista de unidades territoriais
function abreJanelaListaUnidadesTerritoriais(codTabl, codNivt, idioma) {
    var w;
    w = window.open("/LisUnitTabAPI.aspx?c=" + codTabl + "&n=" + codNivt + "&i=" + idioma, "LisUnitTabAPI", "scrollbars=yes,resizable=yes,width=450,height=490,left=428,top=1");
    w.focus();
}

//-->
</script>
</head>

<body>
<form method="post" action="./desctabapi.aspx?c=5457" id="form1">
<input type="hidden" name="__VIEWSTATE" id="__VIEWSTATE" value="HUH685yZlRDcV6KEAOP4FTUCzFOuUrh62HTpXZnNtBLJv4erkpUFuZfv7jIlRPI8DpDme6CVY/rLjZrI2QaxDvzOK

In [0]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response_ibge.text, "html.parser")

for tag in soup.find_all(["option", "a", "td", "span"]):
    texto = tag.get_text(" ", strip=True)

    if any(termo in texto for termo in [
        "Área plantada",
        "Área colhida",
        "Quantidade produzida",
        "Rendimento médio",
        "Valor da produção"
    ]):
        print(tag)
        print("-" * 100)

<td><span class="tituloTabelaDesc" id="lblNomeTabela">Área plantada ou destinada à colheita, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias e permanentes</span></td>
----------------------------------------------------------------------------------------------------
<span class="tituloTabelaDesc" id="lblNomeTabela">Área plantada ou destinada à colheita, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias e permanentes</span>
----------------------------------------------------------------------------------------------------
<td>
<span id="lstVariaveis_lblIdVariavel_0" style="color:Red">8331</span>  <span id="lstVariaveis_lblNomeVariavel_0">Área plantada ou destinada à colheita (Hectares) [1988 a 2025] - casas decimais:  padrão = 0, máximo = 0</span>
</td>
----------------------------------------------------------------------------------------------------
<span id="lstVariaveis_lblNomeVariav

In [0]:
import requests

# Consulta de validação - PAM/IBGE (Tabela 5457)
# Produto: Soja
# Nível territorial: Unidades da Federação
# Período: 2023

url_ibge = (
    "https://apisidra.ibge.gov.br/values/"
    "t/5457/"
    "n3/all/"
    "v/all/"
    "p/2015-2023/"
    "c782/40124"
)

response_ibge = requests.get(url_ibge, timeout=30)

print("Status:", response_ibge.status_code)
print("Resposta:", response_ibge.text[:2000])

Status: 200
Resposta: [
  {
    "NC": "Nível Territorial (Código)",
    "NN": "Nível Territorial",
    "MC": "Unidade de Medida (Código)",
    "MN": "Unidade de Medida",
    "V": "Valor",
    "D1C": "Unidade da Federação (Código)",
    "D1N": "Unidade da Federação",
    "D2C": "Variável (Código)",
    "D2N": "Variável",
    "D3C": "Ano (Código)",
    "D3N": "Ano",
    "D4C": "Produto das lavouras temporárias e permanentes (Código)",
    "D4N": "Produto das lavouras temporárias e permanentes"
  },
  {
    "NC": "3",
    "NN": "Unidade da Federação",
    "MC": "1006",
    "MN": "Hectares",
    "V": "233605",
    "D1C": "11",
    "D1N": "Rondônia",
    "D2C": "8331",
    "D2N": "Área plantada ou destinada à colheita",
    "D3C": "2015",
    "D3N": "2015",
    "D4C": "40124",
    "D4N": "Soja (em grão)"
  },
  {
    "NC": "3",
    "NN": "Unidade da Federação",
    "MC": "1006",
    "MN": "Hectares",
    "V": "246171",
    "D1C": "11",
    "D1N": "Rondônia",
    "D2C": "8331",
    "D2N": "Á

In [0]:
# Converte a resposta da API SIDRA/IBGE para uma estrutura Python
dados_ibge = response_ibge.json()

# O primeiro registro contém apenas a descrição das colunas
dados_ibge = dados_ibge[1:]

# Cria DataFrame Spark
df_ibge_bronze = spark.createDataFrame(dados_ibge)

print(f"Quantidade de registros: {df_ibge_bronze.count()}")

display(df_ibge_bronze)

Quantidade de registros: 1944


D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N,MC,MN,NC,NN,V
11,Rondônia,8331,Área plantada ou destinada à colheita,2015,2015,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,233605
11,Rondônia,8331,Área plantada ou destinada à colheita,2016,2016,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,246171
11,Rondônia,8331,Área plantada ou destinada à colheita,2017,2017,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,280103
11,Rondônia,8331,Área plantada ou destinada à colheita,2018,2018,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,300816
11,Rondônia,8331,Área plantada ou destinada à colheita,2019,2019,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,344551
11,Rondônia,8331,Área plantada ou destinada à colheita,2020,2020,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,392642
11,Rondônia,8331,Área plantada ou destinada à colheita,2021,2021,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,400459
11,Rondônia,8331,Área plantada ou destinada à colheita,2022,2022,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,489526
11,Rondônia,8331,Área plantada ou destinada à colheita,2023,2023,40124,Soja (em grão),1006,Hectares,3,Unidade da Federação,589983
11,Rondônia,1008331,Área plantada ou destinada à colheita - percentual do total geral,2015,2015,40124,Soja (em grão),2,Percentual,3,Unidade da Federação,38.07


In [0]:
# Persistência dos dados brutos do IBGE na camada Bronze

df_ibge_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_ibge_pam_soja")

print("Tabela bronze_ibge_pam_soja criada com sucesso.")

Tabela bronze_ibge_pam_soja criada com sucesso.


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM bronze_ibge_pam_soja;

total_registros
1944


In [0]:
%sql
SELECT *
FROM bronze_ibge_pam_soja
LIMIT 20;

D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N,MC,MN,NC,NN,V
27,Alagoas,214,Quantidade produzida,2015,2015,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,550
27,Alagoas,214,Quantidade produzida,2016,2016,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,1043
27,Alagoas,214,Quantidade produzida,2017,2017,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,1240
27,Alagoas,214,Quantidade produzida,2018,2018,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,2475
27,Alagoas,214,Quantidade produzida,2019,2019,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,10363
27,Alagoas,214,Quantidade produzida,2020,2020,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,4640
27,Alagoas,214,Quantidade produzida,2021,2021,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,9391
27,Alagoas,214,Quantidade produzida,2022,2022,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,11242
27,Alagoas,214,Quantidade produzida,2023,2023,40124,Soja (em grão),1017,Toneladas,3,Unidade da Federação,18289
27,Alagoas,112,Rendimento médio da produção,2015,2015,40124,Soja (em grão),33,Quilogramas por Hectare,3,Unidade da Federação,1978


In [0]:
# Fonte ANP - Produção de biodiesel
# Snapshot da base utilizada no desenvolvimento do MVP

url_anp_biodiesel = (
    "https://raw.githubusercontent.com/GenilsonQ/pos-ciencia-dados/"
    "refs/heads/main/mvp-puc/sprint_engenharia_dados/data/"
    "anp_biodiesel_2005_2023_snapshot.csv"
)

response_anp = requests.get(url_anp_biodiesel, timeout=30)

print("Status:", response_anp.status_code)
print("Conteúdo inicial:")
print(response_anp.text[:1000])

Status: 200
Conteúdo inicial:
ano;mes;grande_regiao;unidade_federacao;produtor;produto;producao
2021;FEV;REGIÃO CENTRO-OESTE;MATO GROSSO;AGROSOJA;BIODIESEL;0
2021;JAN;REGIÃO CENTRO-OESTE;MATO GROSSO;AGROSOJA;BIODIESEL;0
2021;DEZ;REGIÃO NORTE;PARÃ;AGROPALMA;BIODIESEL;0
2021;NOV;REGIÃO NORTE;PARÃ;AGROPALMA;BIODIESEL;0
2021;JUN;REGIÃO CENTRO-OESTE;MATO GROSSO;AGROSOJA;BIODIESEL;0
2021;FEV;REGIÃO CENTRO-OESTE;MATO GROSSO;AGRENCO;BIODIESEL;0
2021;JAN;REGIÃO CENTRO-OESTE;MATO GROSSO;AGRENCO;BIODIESEL;0
2021;DEZ;REGIÃO SUL;SANTA CATARINA;ADM (JOAÃABA);BIODIESEL;8124,688
2021;MAR;REGIÃO CENTRO-OESTE;MATO GROSSO;ADM (RONDONOPOLIS);BIODIESEL;30698,563
2021;FEV;REGIÃO CENTRO-OESTE;MATO GROSSO;ADM (RONDONOPOLIS);BIODIESEL;23086,947
2021;JAN;REGIÃO CENTRO-OESTE;MATO GROSSO;ADM (RONDONOPOLIS);BIODIESEL;15839,28
2021;DEZ;REGIÃO SUDESTE;MINAS GERAIS;ABDIESEL;BIODIESEL;0
2021;NOV;REGIÃO SUDESTE;MINAS GERAIS;ABDIESEL;BIODIESEL;0
2021;OUT;REGIÃO SUDESTE;MINAS GERAIS;ABDIESEL;BIODIESEL;0


In [0]:
from io import StringIO
import pandas as pd

# Leitura do CSV retornado pela ANP
df_anp_pandas = pd.read_csv(
    StringIO(response_anp.text),
    sep=";"
)

print("Quantidade de registros:", len(df_anp_pandas))
print("Colunas:", df_anp_pandas.columns.tolist())

display(df_anp_pandas.head(20))

Quantidade de registros: 23864
Colunas: ['ano', 'mes', 'grande_regiao', 'unidade_federacao', 'produtor', 'produto', 'producao']


ano,mes,grande_regiao,unidade_federacao,produtor,produto,producao
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,DEZ,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,NOV,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,JUN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,DEZ,REGIÃO SUL,SANTA CATARINA,ADM (JOAÃABA),BIODIESEL,"8124,688"
2021,MAR,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"30698,563"
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"23086,947"


In [0]:
# Converte o DataFrame Pandas para Spark
df_anp_bronze = spark.createDataFrame(df_anp_pandas)

print(f"Quantidade de registros no DataFrame Spark: {df_anp_bronze.count()}")

display(df_anp_bronze)

Quantidade de registros no DataFrame Spark: 23864


ano,mes,grande_regiao,unidade_federacao,produtor,produto,producao
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,DEZ,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,NOV,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,JUN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,DEZ,REGIÃO SUL,SANTA CATARINA,ADM (JOAÃABA),BIODIESEL,"8124,688"
2021,MAR,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"30698,563"
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"23086,947"


Durante a ingestão dos dados da ANP, os nomes originais das colunas apresentaram caracteres especiais incompatíveis com a persistência padrão em tabelas Delta. Dessa forma, os nomes dos campos foram ajustados, preservando os valores originais dos registros.

In [0]:
# Ajuste técnico dos nomes das colunas para compatibilidade com Delta

df_anp_bronze = (
    df_anp_bronze
    .toDF(
        "ano",
        "mes",
        "grande_regiao",
        "unidade_federacao",
        "produtor",
        "produto",
        "producao"
    )
)

# Persistência na camada Bronze
df_anp_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_anp_biodiesel")

print("Tabela bronze_anp_biodiesel criada com sucesso.")

Tabela bronze_anp_biodiesel criada com sucesso.


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM bronze_anp_biodiesel;

total_registros
23864


In [0]:
%sql
SELECT *
FROM bronze_anp_biodiesel
LIMIT 20;

ano,mes,grande_regiao,unidade_federacao,produtor,produto,producao
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,DEZ,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,NOV,REGIÃO NORTE,PARÃ,AGROPALMA,BIODIESEL,0
2021,JUN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGROSOJA,BIODIESEL,0
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,JAN,REGIÃO CENTRO-OESTE,MATO GROSSO,AGRENCO,BIODIESEL,0
2021,DEZ,REGIÃO SUL,SANTA CATARINA,ADM (JOAÃABA),BIODIESEL,"8124,688"
2021,MAR,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"30698,563"
2021,FEV,REGIÃO CENTRO-OESTE,MATO GROSSO,ADM (RONDONOPOLIS),BIODIESEL,"23086,947"


Na célula 15, vamos repetir a estratégia segura: baixar e apenas inspecionar primeiro (enxergar exatamente encoding, separador e cabeçalho).

In [0]:
# Fonte ANP - Matérias-primas utilizadas na produção de biodiesel
# Snapshot da base utilizada no desenvolvimento do MVP

url_anp_materia_prima = (
    "https://raw.githubusercontent.com/GenilsonQ/pos-ciencia-dados/"
    "refs/heads/main/mvp-puc/sprint_engenharia_dados/data/"
    "anp_materia_prima_2017_2023_snapshot.csv"
)

response_materia_prima = requests.get(
    url_anp_materia_prima,
    timeout=30
)

print("Status:", response_materia_prima.status_code)
print("Conteúdo inicial:")
print(response_materia_prima.text[:1500])

Status: 200
Conteúdo inicial:
mes_ano;regiao;estado;produto;quantidade_m3
06/2022;CENTRO OESTE;Mato Grosso do Sul;ÃLEO DE SOJA (GLYCINE MAX);16490
06/2022;NORDESTE;Bahia;ÃLEO DE SOJA (GLYCINE MAX);26468
06/2022;NORDESTE;PiauÃ­;ÃLEO DE SOJA (GLYCINE MAX);5139
06/2022;NORTE;Tocantins;ÃLEO DE SOJA (GLYCINE MAX);6474
06/2022;SUDESTE;SÃ£o Paulo;ÃLEO DE SOJA (GLYCINE MAX);102
06/2022;SUL;ParanÃ¡;ÃLEO DE SOJA (GLYCINE MAX);45596
06/2022;SUL;Rio Grande do Sul;ÃLEO DE SOJA (GLYCINE MAX);95433
06/2022;SUL;Santa Catarina;ÃLEO DE SOJA (GLYCINE MAX);11901
06/2022;CENTRO OESTE;GoiÃ¡s;OUTROS MATERIAIS GRAXOS;15388
06/2022;CENTRO OESTE;Mato Grosso;OUTROS MATERIAIS GRAXOS;20987
06/2022;NORDESTE;Bahia;OUTROS MATERIAIS GRAXOS;7142
06/2022;NORTE;Tocantins;OUTROS MATERIAIS GRAXOS;844
06/2022;SUDESTE;Minas Gerais;OUTROS MATERIAIS GRAXOS;7352
06/2022;SUDESTE;Rio de Janeiro;OUTROS MATERIAIS GRAXOS;705
06/2022;SUDESTE;SÃ£o Paulo;OUTROS MATERIAIS GRAXOS;551
06/2022;SUL;ParanÃ¡;OUTROS MATERIAIS GRAXOS;15

In [0]:
import pandas as pd
from io import StringIO

# Leitura do CSV de matérias-primas da ANP

df_materia_prima_pandas = pd.read_csv(
    StringIO(response_materia_prima.text),
    sep=";"
)

print("Quantidade de registros:", len(df_materia_prima_pandas))
print("Colunas:", df_materia_prima_pandas.columns.tolist())

display(df_materia_prima_pandas.head(20))

Quantidade de registros: 4745
Colunas: ['mes_ano', 'regiao', 'estado', 'produto', 'quantidade_m3']


mes_ano,regiao,estado,produto,quantidade_m3
06/2022,CENTRO OESTE,Mato Grosso do Sul,ÃLEO DE SOJA (GLYCINE MAX),16490
06/2022,NORDESTE,Bahia,ÃLEO DE SOJA (GLYCINE MAX),26468
06/2022,NORDESTE,PiauÃ­,ÃLEO DE SOJA (GLYCINE MAX),5139
06/2022,NORTE,Tocantins,ÃLEO DE SOJA (GLYCINE MAX),6474
06/2022,SUDESTE,SÃ£o Paulo,ÃLEO DE SOJA (GLYCINE MAX),102
06/2022,SUL,ParanÃ¡,ÃLEO DE SOJA (GLYCINE MAX),45596
06/2022,SUL,Rio Grande do Sul,ÃLEO DE SOJA (GLYCINE MAX),95433
06/2022,SUL,Santa Catarina,ÃLEO DE SOJA (GLYCINE MAX),11901
06/2022,CENTRO OESTE,GoiÃ¡s,OUTROS MATERIAIS GRAXOS,15388
06/2022,CENTRO OESTE,Mato Grosso,OUTROS MATERIAIS GRAXOS,20987


In [0]:
# Converte o DataFrame Pandas para Spark
df_materia_prima_bronze = spark.createDataFrame(df_materia_prima_pandas)

# Ajuste técnico dos nomes das colunas para compatibilidade com Delta
df_materia_prima_bronze = df_materia_prima_bronze.toDF(
    "mes_ano",
    "regiao",
    "estado",
    "produto",
    "quantidade_m3"
)

print(
    f"Quantidade de registros no DataFrame Spark: "
    f"{df_materia_prima_bronze.count()}"
)

display(df_materia_prima_bronze)

Quantidade de registros no DataFrame Spark: 4745


mes_ano,regiao,estado,produto,quantidade_m3
06/2022,CENTRO OESTE,Mato Grosso do Sul,ÃLEO DE SOJA (GLYCINE MAX),16490
06/2022,NORDESTE,Bahia,ÃLEO DE SOJA (GLYCINE MAX),26468
06/2022,NORDESTE,PiauÃ­,ÃLEO DE SOJA (GLYCINE MAX),5139
06/2022,NORTE,Tocantins,ÃLEO DE SOJA (GLYCINE MAX),6474
06/2022,SUDESTE,SÃ£o Paulo,ÃLEO DE SOJA (GLYCINE MAX),102
06/2022,SUL,ParanÃ¡,ÃLEO DE SOJA (GLYCINE MAX),45596
06/2022,SUL,Rio Grande do Sul,ÃLEO DE SOJA (GLYCINE MAX),95433
06/2022,SUL,Santa Catarina,ÃLEO DE SOJA (GLYCINE MAX),11901
06/2022,CENTRO OESTE,GoiÃ¡s,OUTROS MATERIAIS GRAXOS,15388
06/2022,CENTRO OESTE,Mato Grosso,OUTROS MATERIAIS GRAXOS,20987


In [0]:
# Persistência dos dados brutos de matérias-primas da ANP na camada Bronze

df_materia_prima_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_anp_materia_prima")

print("Tabela bronze_anp_materia_prima criada com sucesso.")

Tabela bronze_anp_materia_prima criada com sucesso.


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM bronze_anp_materia_prima;

total_registros
4745


In [0]:
%sql
SELECT *
FROM bronze_anp_materia_prima
LIMIT 20;

mes_ano,regiao,estado,produto,quantidade_m3
08/2018,SUDESTE,SÃ£o Paulo,ÃLEO DE FRITURA USADO,3902
08/2018,SUL,ParanÃ¡,ÃLEO DE FRITURA USADO,118
08/2018,SUL,Rio Grande do Sul,ÃLEO DE FRITURA USADO,123
08/2018,CENTRO OESTE,GoiÃ¡s,ÃLEO DE MILHO,995
08/2018,SUDESTE,SÃ£o Paulo,ÃLEO DE MILHO,39
08/2018,NORDESTE,Bahia,ÃLEO DE PALMA/DENDÃ (ELAEIS GUINEENSIS OU ELAEIS O,3197
08/2018,CENTRO OESTE,GoiÃ¡s,ÃLEO DE SOJA (GLYCINE MAX),58452
08/2018,CENTRO OESTE,Mato Grosso,ÃLEO DE SOJA (GLYCINE MAX),74613
08/2018,CENTRO OESTE,Mato Grosso do Sul,ÃLEO DE SOJA (GLYCINE MAX),13012
08/2018,NORDESTE,Bahia,ÃLEO DE SOJA (GLYCINE MAX),10863


Verificando o intervalo temporal real do arquivo de matérias-primas.

In [0]:
%sql
SELECT
    MIN(TO_DATE(mes_ano, 'MM/yyyy')) AS primeira_competencia,
    MAX(TO_DATE(mes_ano, 'MM/yyyy')) AS ultima_competencia,
    COUNT(DISTINCT mes_ano) AS quantidade_competencias
FROM bronze_anp_materia_prima;

primeira_competencia,ultima_competencia,quantidade_competencias
2017-01-01,2023-08-01,80


In [0]:
df_materia_prima_original = spark.table("bronze_anp_materia_prima")

print("Registros:", df_materia_prima_original.count())
print("Colunas:", df_materia_prima_original.columns)

display(df_materia_prima_original.limit(20))

Registros: 4745
Colunas: ['mes_ano', 'regiao', 'estado', 'produto', 'quantidade_m3']


mes_ano,regiao,estado,produto,quantidade_m3
08/2023,CENTRO OESTE,GoiÃ¡s,ÃCIDO GRAXO DE ÃLEO DE SOJA,652
08/2023,CENTRO OESTE,Mato Grosso,ÃCIDO GRAXO DE ÃLEO DE SOJA,956
08/2023,SUDESTE,SÃ£o Paulo,ÃCIDO GRAXO DE ÃLEO DE SOJA,511
08/2023,CENTRO OESTE,GoiÃ¡s,GORDURA BOVINA,6595
08/2023,CENTRO OESTE,Mato Grosso,GORDURA BOVINA,447
08/2023,NORDESTE,Bahia,GORDURA BOVINA,11416
08/2023,NORTE,ParÃ¡,GORDURA BOVINA,1208
08/2023,NORTE,RondÃ´nia,GORDURA BOVINA,3800
08/2023,SUDESTE,Minas Gerais,GORDURA BOVINA,1502
08/2023,SUL,ParanÃ¡,GORDURA BOVINA,745
